<a href="https://colab.research.google.com/github/satishmathapa/mac247-labs/blob/main/lab1/MAC247_LAB1_S03_Access_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAC 247 Graded Lab 1, Part 2: reviewing a roster against the model

**Part 2, Thursday, September 24, 2026. Google Colab in a browser, on a B124 machine or your own
laptop. Every cell below uses only the Python 3 standard library, so there is nothing to install.**

In Part 1 (`MAC247_LAB1_S01_Reference_Monitor.ipynb`) you wrote the model: what each role
legitimately needs, and which duties must never sit with one person. Today the model does the
only thing a model is for, which is to disagree with something.

This notebook builds the roster and writes the two files a Linux system would have exported from
its own account database, `identities.csv` and `entitlements.csv`. It then reads those files back
as files, not as the variables that produced them, and reviews them: every entitlement an account
holds that its role does not require, every pair of individually reasonable entitlements that
arrived at the same person, and every identity nobody has touched in months. Then it checks
itself against the assignment card, and turns each finding into one command.

**Nothing in this notebook contacts any host outside it, and nothing here needs root.** Every
row it analyzes is a row it generated from your own student ID.

## Section 1. Your token, the model, and your assignment card

Use the **same student ID** you used in Part 1 on Tuesday. The card is derived from it, so the
cell below recomputes the card Part 1 printed, and you do not have to carry anything between the
two parts. Your token carries the time you ran the cell as well, so it will not match Part 1's,
and that is expected.

In [1]:
import hashlib, datetime, csv, io, os

sid = '24647180'
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC247 Lab 1 token:', TOKEN)

# ---- the model, exactly as Part 1 stated it -------------------------------------------
ROLE_REQUIRED = {
    'finance':  {'group:mac247fin'},
    'analyst':  {'group:mac247ana'},
    'auditor':  {'group:mac247aud'},
    'backup':   {'group:mac247bkp'},
}
ROSTER = {
    'finance':  'mac247_finlead',
    'analyst':  'mac247_analyst',
    'auditor':  'mac247_auditor',
    'backup':   'mac247_backup',
}
SOD_CONFLICTS = [
    ({'group:mac247fin'}, {'group:mac247aud'},
     'posting a transaction and signing off the audit of it'),
    ({'group:mac247bkp'}, {'group:mac247aud'},
     'holding the backups and attesting that the backups are intact'),
    ({'sudo:ALL'},        {'group:mac247aud'},
     'unrestricted administration and custody of the audit record'),
]
ROLE_OF = {user: role for role, user in ROSTER.items()}

# ---- the assignment card, recomputed from the same student ID ----------------------------
EXCESS_POOL = ['group:mac247fin', 'group:mac247aud', 'group:mac247bkp', 'sudo:ALL']

def conflicting_additions(role):
    held = ROLE_REQUIRED[role]
    out = []
    for e in EXCESS_POOL:
        if e in held:
            continue
        for a, b, _ in SOD_CONFLICTS:
            if (a <= held and b <= {e}) or (b <= held and a <= {e}):
                out.append(e)
                break
    return out

h = int(hashlib.sha256((sid + '-lab1-card').encode()).hexdigest(), 16)
roles = sorted(ROLE_REQUIRED)
candidates = [r for r in roles if conflicting_additions(r)]
excess_role = candidates[h % len(candidates)]
choices = conflicting_additions(excess_role)
CARD = {
    'fault_1_over_entitled_account': ROSTER[excess_role],
    'fault_1_extra_entitlement': choices[(h // 7) % len(choices)],
    'fault_2_stale_account': ROSTER[roles[(h // 31) % len(roles)]],
}
for k, v in CARD.items():
    print(f'{k:32s} {v}')

assert CARD['fault_1_extra_entitlement'] not in ROLE_REQUIRED[excess_role]
print('\n  CHECK PASS: the card recomputed from your student ID, and the model is loaded')

MAC247 Lab 1 token: 9f6b04ecf893d37f
fault_1_over_entitled_account    mac247_auditor
fault_1_extra_entitlement        group:mac247fin
fault_2_stale_account            mac247_backup

  CHECK PASS: the card recomputed from your student ID, and the model is loaded


## Section 2. The export this notebook writes

A real Linux system does not hand you a review. It hands you its own account database, and
somebody has to turn that into two files with agreed columns before any analysis can begin. The
cell below writes those two files the way `pwd`, `grp` and `chage` would have written them:

- `identities.csv`, one row per account: who it is, what shell it gets, whether it is locked, and
  when its password last changed.
- `entitlements.csv`, one row per entitlement held, produced by walking the groups rather than
  the accounts, which is why the rows are not grouped by user. Real exports look like this, and
  an analysis that assumes otherwise breaks on the first real file it sees.

The two faults from your card are in there. Everything else is what a correctly provisioned
roster looks like.

In [2]:
TODAY = datetime.date.today()

SHELL = {'mac247_backup': '/usr/sbin/nologin'}      # a service account never logs in
UID0 = 2001

identity_rows, entitlement_rows = [], []
for i, role in enumerate(sorted(ROSTER)):
    user = ROSTER[role]
    stale = (user == CARD['fault_2_stale_account'])
    identity_rows.append({
        'username': user,
        'uid': str(UID0 + i),
        'primary_group': sorted(ROLE_REQUIRED[role])[0].split(':', 1)[1],
        'shell': SHELL.get(user, '/bin/bash'),
        'locked': 'no',
        'last_change': (TODAY - datetime.timedelta(days=430 if stale else 21)).isoformat(),
    })
    for e in sorted(ROLE_REQUIRED[role]):
        entitlement_rows.append({'username': user, 'entitlement': e, 'source': '/etc/group'})

extra = CARD['fault_1_extra_entitlement']
entitlement_rows.append({
    'username': CARD['fault_1_over_entitled_account'],
    'entitlement': extra,
    'source': '/etc/group' if extra.startswith('group:')
              else '/etc/sudoers.d/mac247_' + CARD['fault_1_over_entitled_account'],
})
# a real export walks /etc/group, so the rows come out ordered by entitlement, not by account
entitlement_rows.sort(key=lambda r: (r['entitlement'], r['username']))

def write_csv(path, columns, rows):
    with open(path, 'w', newline='') as fh:
        w = csv.DictWriter(fh, columns)
        w.writeheader()
        w.writerows(rows)
    print('==', path, '=' * (56 - len(path)))
    print(open(path).read())

write_csv('identities.csv',
          ['username', 'uid', 'primary_group', 'shell', 'locked', 'last_change'],
          identity_rows)
write_csv('entitlements.csv', ['username', 'entitlement', 'source'], entitlement_rows)
print('  CHECK PASS: two export files written, '
      f'{len(identity_rows)} identities and {len(entitlement_rows)} entitlement rows')

== identities.csv ==========================================
username,uid,primary_group,shell,locked,last_change
mac247_analyst,2001,mac247ana,/bin/bash,no,2026-09-05
mac247_auditor,2002,mac247aud,/bin/bash,no,2026-09-05
mac247_backup,2003,mac247bkp,/usr/sbin/nologin,no,2025-07-23
mac247_finlead,2004,mac247fin,/bin/bash,no,2026-09-05

== entitlements.csv ========================================
username,entitlement,source
mac247_analyst,group:mac247ana,/etc/group
mac247_auditor,group:mac247aud,/etc/group
mac247_backup,group:mac247bkp,/etc/group
mac247_auditor,group:mac247fin,/etc/group
mac247_finlead,group:mac247fin,/etc/group

  CHECK PASS: two export files written, 4 identities and 5 entitlement rows


### Reading them back as files

From here on the analysis sees the files and not the variables that made them, which is the only
honest way to test a review: a detector that reads the structure it built itself proves nothing.
The loader checks the header before it reads a single row. A loader that silently accepts a file
whose columns are in a different order will read the shell out of the locked column and report
nonsense with complete confidence.

In [3]:
def load(path, expected):
    rows = list(csv.DictReader(open(path)))
    assert rows, f'{path} has a header and no rows'
    assert list(rows[0].keys()) == expected, (
        f'{path} header is {list(rows[0].keys())}, expected {expected}')
    return rows

IDENTITIES = load('identities.csv',
                  ['username', 'uid', 'primary_group', 'shell', 'locked', 'last_change'])
ENTITLEMENTS = load('entitlements.csv', ['username', 'entitlement', 'source'])

held = {}
for r in ENTITLEMENTS:
    held.setdefault(r['username'], set()).add(r['entitlement'])

print(f'{"account":18s} {"uid":5s} {"shell":20s} {"last change":12s} entitlements held')
for r in IDENTITIES:
    u = r['username']
    print(f'{u:18s} {r["uid"]:5s} {r["shell"]:20s} {r["last_change"]:12s} '
          f'{", ".join(sorted(held.get(u, {"none"})))}')
print('\n  CHECK PASS: both files parsed, headers match, '
      f'{len(IDENTITIES)} accounts and {len(ENTITLEMENTS)} entitlement rows loaded')

account            uid   shell                last change  entitlements held
mac247_analyst     2001  /bin/bash            2026-09-05   group:mac247ana
mac247_auditor     2002  /bin/bash            2026-09-05   group:mac247aud, group:mac247fin
mac247_backup      2003  /usr/sbin/nologin    2025-07-23   group:mac247bkp
mac247_finlead     2004  /bin/bash            2026-09-05   group:mac247fin

  CHECK PASS: both files parsed, headers match, 4 accounts and 5 entitlement rows loaded


## Section 3. The review

Three questions, asked of the population rather than of one account at a time. That is the whole
reason a review exists: every one of these findings is invisible when you look at a single
account, because nothing about a single account looks wrong.

### Least privilege, measured

An account is over-entitled when it holds something its role does not require. That sentence is
only checkable because Part 1 wrote down what each role requires before anybody had an account
to defend. Writing the required set *after* looking at the accounts is how an organization ends
up certifying that everyone has exactly what they have.

In [4]:
excess = []
for user in sorted(held):
    role = ROLE_OF.get(user)
    if role is None:
        excess.append((user, 'UNKNOWN_ACCOUNT', 'not on the roster at all'))
        continue
    for e in sorted(held[user] - ROLE_REQUIRED[role]):
        excess.append((user, e, f'role {role} requires only {sorted(ROLE_REQUIRED[role])}'))

print(f'{"account":18s} {"excess entitlement":22s} why it is excess')
for u, e, why in excess:
    print(f'{u:18s} {e:22s} {why}')
if not excess:
    print('  (none)')

assert len(held) == len(ROSTER), 'every roster account should appear in the export'
print(f'\n  CHECK PASS: {len(held)} accounts compared against the role model, '
      f'{len(excess)} over-entitlement(s) found')

account            excess entitlement     why it is excess
mac247_auditor     group:mac247fin        role auditor requires only ['group:mac247aud']

  CHECK PASS: 4 accounts compared against the role model, 1 over-entitlement(s) found


### Separation of duties, measured

A separation-of-duties break is not one bad entitlement. It is a pair of individually reasonable
entitlements that arrived at the same person, usually months apart, usually because somebody
changed job and kept the old access. Nothing about either entitlement looks wrong on its own,
which is exactly why this needs a population-level check rather than an eye.

In [5]:
sod = []
for user in sorted(held):
    for side_a, side_b, why in SOD_CONFLICTS:
        if side_a <= held[user] and side_b <= held[user]:
            sod.append((user, sorted(side_a)[0], sorted(side_b)[0], why))

for u, a, b, why in sod:
    print(f'CONFLICT  {u}')
    print(f'          holds {a} and {b}')
    print(f'          which together allow {why}\n')
if not sod:
    print('no separation-of-duties conflicts in this export')

checked = len(held) * len(SOD_CONFLICTS)
print(f'  CHECK PASS: {checked} account-and-conflict pairs tested, {len(sod)} break(s) found')

CONFLICT  mac247_auditor
          holds group:mac247fin and group:mac247aud
          which together allow posting a transaction and signing off the audit of it

  CHECK PASS: 12 account-and-conflict pairs tested, 1 break(s) found


### Stale identities

The identity life cycle is proofing, provisioning, maintenance and entitlement, and the stage
organizations skip is the review. An account nobody has touched in over a year is not evidence of
an attack. It is evidence that nobody is looking, which is the condition an attacker needs rather
than the attack itself.

In [6]:
STALE_DAYS = 180

stale = []
for r in IDENTITIES:
    age = (TODAY - datetime.date.fromisoformat(r['last_change'])).days
    if age > STALE_DAYS or r['locked'].lower() in ('yes', 'true', 'l'):
        stale.append((r['username'], age, r['locked']))

for u, age, locked in stale:
    print(f'{u:18s} password last changed {age:4d} days ago, locked={locked}')
if not stale:
    print(f'no account has gone more than {STALE_DAYS} days without a password change')

print(f'\n  CHECK PASS: {len(IDENTITIES)} identities aged against a '
      f'{STALE_DAYS}-day threshold, {len(stale)} stale')

mac247_backup      password last changed  430 days ago, locked=no

  CHECK PASS: 4 identities aged against a 180-day threshold, 1 stale


## Section 4. Did the review find what was planted?

This is the check that separates a detector from a plausible-looking report. You knew before the
export was written what was wrong with it. If either line below does not read found, the detector
is not detecting, and finding that out is worth more to you than a clean run.

In [7]:
excess_hits = [(u, e) for u, e, _ in excess
               if u == CARD['fault_1_over_entitled_account']
               and e == CARD['fault_1_extra_entitlement']]
sod_hits = [u for u, _, _, _ in sod if u == CARD['fault_1_over_entitled_account']]
stale_hits = [u for u, _, _ in stale if u == CARD['fault_2_stale_account']]

print(f'fault 1, {CARD["fault_1_extra_entitlement"]} on '
      f'{CARD["fault_1_over_entitled_account"]:18s} -> '
      f'{"found" if excess_hits else "NOT FOUND"} as an over-entitlement, '
      f'{"found" if sod_hits else "NOT FOUND"} as a separation-of-duties break')
print(f'fault 2, stale account {CARD["fault_2_stale_account"]:18s} -> '
      f'{"found" if stale_hits else "NOT FOUND"}')

assert excess_hits and sod_hits and stale_hits, 'the review missed a fault it was told about'
print('\n  CHECK PASS: both planted faults were recovered from the exported files')
print('  token:', TOKEN)

fault 1, group:mac247fin on mac247_auditor     -> found as an over-entitlement, found as a separation-of-duties break
fault 2, stale account mac247_backup      -> found

  CHECK PASS: both planted faults were recovered from the exported files
  token: 9f6b04ecf893d37f


## Section 5. The remediation plan

An analysis that ends in a finding is half a control. The other half is the change, and the change
has to be specific enough that somebody else could make it without asking you what you meant.

The cell below turns every finding into one command against the system that produced this export,
and names the NIST SP 800-53 Rev 5 control the change satisfies. Check every control number
against the catalog before you put it in your report: a control identifier you did not verify is
worse than none, because it looks authoritative.

In [8]:
REMEDIATION = {
    'group': ('sudo gpasswd -d {user} {name}',
              'AC-6 Least Privilege'),
    'sudo':  ('sudo rm -f /etc/sudoers.d/mac247_{user}',
              'AC-6(5) Least Privilege, Privileged Accounts'),
    'stale': ('sudo usermod -L {user} && sudo chage -E 0 {user}',
              'AC-2(3) Account Management, Disable Accounts'),
}

plan = []
for u, e, _ in excess:
    kind = 'group' if e.startswith('group:') else 'sudo'
    cmd, control = REMEDIATION[kind]
    plan.append((cmd.format(user=u, name=e.split(':', 1)[1]), control,
                 f'{u} holds {e}, which its role does not require'))
for u, age, _ in stale:
    cmd, control = REMEDIATION['stale']
    plan.append((cmd.format(user=u), control,
                 f'{u} has not changed its password in {age} days'))

print('# MAC 247 Lab 1 remediation plan, token', TOKEN)
for cmd, control, why in plan:
    print(f'\n# {why}\n# control: {control}\n{cmd}')

assert plan, 'no findings, so no plan: check that the export was written before this ran'
print(f'\n  CHECK PASS: {len(plan)} command(s) generated, each tied to one finding '
      'and one control')

# MAC 247 Lab 1 remediation plan, token 9f6b04ecf893d37f

# mac247_auditor holds group:mac247fin, which its role does not require
# control: AC-6 Least Privilege
sudo gpasswd -d mac247_auditor mac247fin

# mac247_backup has not changed its password in 430 days
# control: AC-2(3) Account Management, Disable Accounts
sudo usermod -L mac247_backup && sudo chage -E 0 mac247_backup

  CHECK PASS: 2 command(s) generated, each tied to one finding and one control


## Closing Part 2, and closing Lab 1

Eight `CHECK PASS` lines must appear above. Then, about 20 minutes before class ends:

1. **File, then Save a copy in GitHub.** Choose `mac247-labs`, path
   `lab1/MAC247_LAB1_S03_Access_Review.ipynb`, and use `lab1 part 2` as the commit message.
2. Screenshots, both showing your Google account name and your token:
   `lab1_s03_review.png`, the over-entitlements, the conflict and the two ground-truth lines;
   and `lab1_s03_remediation_plan.png`, the generated command block. Upload both to `lab1` on
   github.com with Add file, then Upload files.
3. Open the commit history on github.com, confirm `lab1` holds both notebooks and every
   screenshot, then open your newest commit and copy its address onto the first page of your
   report.
4. Sign out: Runtime, then Disconnect and delete runtime, and on a shared computer also sign out
   of Google and GitHub in the browser. Capture `lab1_s03_logout.png` for your report.

`identities.csv` and `entitlements.csv` are written into a disposable runtime, so commit them if
you like, but the copy that matters is the one printed into this notebook's output.

## What goes in the report

1. The review found an over-entitlement and a separation-of-duties break on the same account.
   Explain why those are two findings and not one, in terms of what each one would cost to fix
   and what each one would let somebody do.
2. Nothing about the over-entitled account looks wrong when you read its row on its own. Say
   which of the two other files, or which additional column, is what makes the finding visible,
   and why an account-by-account review would have missed it.
3. The stale account was not locked and had not been touched in over a year. Name the stage of
   the identity life cycle that failed, and say what the account still owned on the system after
   somebody stopped using it.
4. The plan names a control number for every command. Pick one of them, look it up in the NIST
   SP 800-53 Rev 5 catalog, and write two sentences: what the control actually requires, and
   which command in your plan is the evidence that it is satisfied.
5. Your 150-250 word reflection paragraph, on one idea in Lab 1 you did not previously
   understand and why it matters.

**Submit by Sunday, September 27, 2026, 11:59 PM**, in the Lab 1 assignment folder in
Brightspace: only the report, as one PDF named `MAC247_LAB1_YourLastName.pdf`, with the commit
URL on its first page.